# Beginner tour 0: Check your setup

Run this notebook **first**, before any other. It checks everything the tour needs, one
thing at a time, and if something is wrong it tells you **exactly how to fix it**. It takes
about two minutes. If every check says **OK**, you're ready for notebook 1.

**You don't need to understand the code in this notebook yet.** Just run it and read what it
prints.

### Running a notebook

* A notebook is made of **cells**. Grey boxes are code; text like this is an explanation.
* Click a code cell and press **Shift + Enter** to run it. The result appears below it, and
  the cursor moves to the next cell.
* While a cell runs you'll see `[*]` (or a spinning icon) next to it. Wait until it finishes.
* If something goes very wrong: in the top bar, **Restart** the kernel, then run the cells
  again from the top.

---
## Check 1: the right Python

The tour needs the **py310** environment, a ready-made set of Python and its add-on
packages. In VS Code you choose it with the **kernel picker** in the top-right corner of the
notebook: click it, choose *Python Environments*, then **py310**
(`/N/slate/mengjing/miniconda3/envs/py310`).

In [ ]:
import sys
problems = []          # we'll collect anything that's wrong here

def report(name, ok, fix=""):
    # print one line per check: OK, or PROBLEM plus how to fix it
    if ok:
        print(f"OK       {name}")
    else:
        print(f"PROBLEM  {name}\n         how to fix: {fix}")
        problems.append(name)

report(f"Python {sys.version.split()[0]} at {sys.executable}",
       "envs/py310" in sys.executable,
       "pick the py310 kernel in the top-right corner, then run this cell again")

---
## Check 2: the add-on packages

Python on its own does little. These **packages** (toolboxes other people wrote) do the
heavy lifting: tables, numbers, AI models, chemistry, charts.

In [ ]:
import importlib
needed = {
    "numpy": "lists of numbers (arrays)",
    "pandas": "tables",
    "matplotlib": "charts",
    "torch": "PyTorch: neural networks",
    "sklearn": "classic machine learning (random forests)",
    "rdkit": "chemistry: reading and drawing molecules",
}
for package, job in needed.items():
    try:
        module = importlib.import_module(package)
        report(f"{package:11s} {getattr(module, '__version__', ''):10s} ({job})", True)
    except ImportError:
        report(f"{package} ({job})", False, "you're probably on the wrong kernel; see check 1")

---
## Check 3: a GPU (graphics card)

Most of the tour runs on a normal processor (CPU). A **GPU** makes the neural networks much
faster, and notebook **4b** needs one for its graph network. This computer has two.

In [ ]:
import torch
if torch.cuda.is_available():
    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    report(f"GPU: {', '.join(names)}", True)
else:
    print("NOTE     no GPU found. Notebooks 1-8 still run (a bit slower). Skip the MPN part of notebook 4b.")

---
## Check 4: the project's code

The tour runs the real code of the **al-molecular** research project, from a **frozen copy**
that doesn't change (the live project changes almost every day, and the lessons quote its
code line by line). The copy is a folder next to this course:

In [ ]:
from pathlib import Path
import subprocess

AL_REPO = Path("/N/slate/mengjing/repos/al-molecular-tutorial")
report(f"frozen copy at {AL_REPO}", (AL_REPO / "run_experiment.py").exists(),
       "ask whoever set up the course to run the 'Set it up' step in learn-ai-research/README.md")

if AL_REPO.exists():
    version = subprocess.run(["git", "-C", str(AL_REPO), "describe", "--tags"], capture_output=True, text=True).stdout.strip()
    changed = subprocess.run(["git", "-C", str(AL_REPO), "status", "--short", "--untracked-files=no"],
                             capture_output=True, text=True).stdout.strip()
    report(f"it's the tour's version: {version or 'unknown'}", version.startswith("tutorial-"),
           "in a terminal: cd " + str(AL_REPO) + " && git checkout tutorial-2026-10-20")
    report("nobody has edited it", changed == "",
           "someone changed files in the frozen copy, so lessons may not match. To undo every change: "
           f"git -C {AL_REPO} checkout -- .")
    report("its saved experiment results are there", (AL_REPO / "runs" / "fusionfix_greedy_frac0.001_seed42").exists(),
           "copy the runs as described in learn-ai-research/README.md ('Set it up')")

**Please don't edit files inside the frozen copy.** If you want to experiment with the
project's code, for example putting a `breakpoint()` in it while learning to debug, make
your change, then undo it afterwards with the `git ... checkout -- .` command above. Or use
VS Code's debugger, which sets breakpoints (the red dots) without changing any file.

---
## Check 5: the tour's data

The tour uses a small copy of real data: 21,044 molecules with their real docking scores
and descriptions. It has to be built once, which takes under a minute. This cell builds it
if it's missing.

In [ ]:
MINI_DIR = Path("..").resolve().parent / "data" / "enhits_mini"      # learn-ai-research/data/enhits_mini
files = ["scores.csv.gz", "molformer_embeddings.npy", "unimol2_embeddings.npy", "mhgged_embeddings.npy"]

if not all((MINI_DIR / f).exists() for f in files):
    print("building the mini dataset (once, under a minute)...")
    sys.path.insert(0, ".")
    import build_mini_dataset
    build_mini_dataset.main()

report(f"mini dataset at {MINI_DIR}", all((MINI_DIR / f).exists() for f in files),
       "the build failed; read the error above. The original data lives on /N/project, so you must be on the IU cluster")

---
## Check 6: everything working together

The real test: load the data with the tour's helper, import the project's main file, and
draw a molecule.

In [ ]:
from tour import *                    # the tour's small helper file (tour.py, in this folder)
import run_experiment as rx

scores, embeddings = load_mini()
report(f"loaded {len(scores):,} molecules and {len(embeddings)} kinds of embedding", len(scores) == 21044)
report(f"the project's code comes from the frozen copy: {rx.__file__}", "al-molecular-tutorial" in rx.__file__,
       "restart the kernel and run this notebook again from the top")
best = scores.nsmallest(1, "score")
draw(best["smiles"], labels=[f"the best molecule: {best['score'].iloc[0]} kcal/mol"], per_row=1)

---
## Result

In [ ]:
if problems:
    print(f"{len(problems)} problem(s) to fix first:")
    for p in problems:
        print("  -", p)
else:
    print("All checks passed. You're ready: open 1_meet_the_data.ipynb")

### Things that look scary but are fine

* **Red text** under a cell is often just a *warning* from a package ("deprecated", "tqdm",
  "ipywidgets"...). If the cell's result appeared, it's fine. Only an **error** (a message
  ending in something like `KeyError` or `ValueError`) means the cell failed.
* **A cell that runs for minutes.** Some cells run full experiments. The notebook says how
  long to expect. Wait for `[*]` to turn into a number.
* **Numbers slightly different from the text.** Results that involve randomness or a GPU can
  differ a little between runs. The text describes the pattern, not the exact digits.

**Stuck?** See `../START_HERE.md` ("When something goes wrong"), and `../GLOSSARY.md` for any
unfamiliar word.